# Phase E: validation-only enhancement screening
Run in Colab after syncing the project to Drive. All candidate runs use seed 42 and validation metrics only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys
import pandas as pd
import yaml
project_root = Path('/content/drive/MyDrive/mizan-arabert-sentiment')
os.chdir(project_root)
sys.path.insert(0, str(project_root))

In [ ]:
%pip install -q -r requirements-bonus.txt

In [ ]:
from src.bonus.screening import (
    CANDIDATES, BASE_DIR, PHASE_D_SUMMARY, screen_candidates,
    check_baseline_equivalence, check_candidate_configs,
    check_loss_and_init_candidates, check_model_candidates,
)
from src.bonus.train import run_training
CHECKS_ENABLED = False

In [ ]:
if CHECKS_ENABLED:
    check_baseline_equivalence()
    check_candidate_configs()
    check_loss_and_init_candidates()
    check_model_candidates()
    print('Phase E checks passed.')

## Optional smoke runs
Smoke artifacts use separate directories and cannot be resumed by real screening runs.

In [ ]:
RUN_SMOKE = True
SMOKE_CANDIDATES = {name: True for name in CANDIDATES}
if RUN_SMOKE:
    for candidate, config_path in CANDIDATES.items():
        if not SMOKE_CANDIDATES[candidate]:
            continue
        smoke_dir = Path('checkpoints/bonus/enh_smoke') / candidate
        run_training(config_path, output_dir=smoke_dir, max_train_samples=200, max_validation_samples=200, epochs=1, resume=(smoke_dir / 'checkpoint_last.pt').is_file())

## Seed 42 screening runs

In [ ]:
RUN_CANDIDATES = {name: False for name in CANDIDATES}
for candidate, config_path in CANDIDATES.items():
    if not RUN_CANDIDATES[candidate]:
        print(f'{candidate}: skipped by switch')
        continue
    with config_path.open(encoding='utf-8') as handle:
        candidate_config = yaml.safe_load(handle)
    run_dir = Path(candidate_config['paths']['output_dir'])
    history_path = run_dir / 'history.csv'
    checkpoint = run_dir / 'checkpoint_last.pt'
    if history_path.is_file() and checkpoint.is_file():
        history = pd.read_csv(history_path)
        complete = ('epoch' in history.columns and len(history) == 10 and sorted(history['epoch'].astype(int).tolist()) == list(range(1, 11)))
        if complete:
            print(f'{candidate}: complete; skipping')
            continue
    config_path_colab = Path(f'/content/enh_{candidate}_seed42.yaml')
    config_path_colab.write_text(yaml.safe_dump(candidate_config, sort_keys=False), encoding='utf-8')
    run_training(config_path_colab, output_dir=run_dir, resume=checkpoint.is_file())

In [ ]:
baseline_history_exists = (BASE_DIR / 'history.csv').is_file()
phase_d_summary_exists = PHASE_D_SUMMARY.is_file()
if baseline_history_exists and phase_d_summary_exists:
    screening_result = screen_candidates()
    print(screening_result['decision'])
else:
    print('Screening deferred: baseline seed 42 history and Phase D summary are both required.')